# D-01 EDA - Migrants by Place of Birth (Census 2011)

**File:** `DS-0000-D01-MDDS.XLSX`  
**What this has:** For each destination state / UT, this table shows where migrants were born, along with total, male, female, rural, and urban counts.

This notebook rebuilds the state-wise `D01_cleaned.csv` used by the dashboard.


In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

INPUT_FILE = 'DS-0000-D01-MDDS.XLSX'
SKIP_ROWS = 4

COL_NAMES = [
    'TableName', 'StateCode', 'DistrictCode', 'AreaName', 'BirthPlace',
    'Total_Persons', 'Total_Males', 'Total_Females',
    'Rural_Persons', 'Rural_Males', 'Rural_Females',
    'Urban_Persons', 'Urban_Males', 'Urban_Females',
]

df_raw = pd.read_excel(INPUT_FILE, skiprows=SKIP_ROWS, header=None)
df_raw.columns = COL_NAMES[:df_raw.shape[1]]
print(f'Raw shape: {df_raw.shape}')
df_raw.head(8)


Raw shape: (3169, 14)


,TableName,StateCode,DistrictCode,AreaName,BirthPlace,Total_Persons,Total_Males,Total_Females,Rural_Persons,Rural_Males,Rural_Females,Urban_Persons,Urban_Males,Urban_Females
0,NaN,NaN,NaN,NaN,1,2,3,4,5,6,7,8,9,10
1,D0101,0.0,0.0,INDIA,Total Population,1210854977,623270258,587584719,833748852,427781058,405967794,377106125,195489200,181616925
2,D0101,0.0,0.0,INDIA,Born within India,1205201066,620582802,584618264,831175009,426664183,404510826,374026057,193918619,180107438
3,D0101,0.0,0.0,INDIA,Within the state of enumeration,1148903503,595735238,553168265,815096974,422199650,392897324,333806529,173535588,160270941
4,D0101,0.0,0.0,INDIA,Born in the place of enumeration,763567558,480479075,283088483,560572499,364822626,195749873,202995059,115656449,87338610
5,D0101,0.0,0.0,INDIA,Born elsewhere in the district of enumeration,264108665,77306410,186802255,193992402,45827679,148164723,70116263,31478731,38637532
6,D0101,0.0,0.0,INDIA,Born in other districts of the state,121227280,37949753,83277527,60532073,11549345,48982728,60695207,26400408,34294799
7,D0101,0.0,0.0,INDIA,States in India beyond the state of enumeration,56297563,24847564,31449999,16078035,4464533,11613502,40219528,20383031,19836497


## 1. Raw snapshot


In [4]:
print(df_raw.dtypes,"\n")
print(df_raw.isnull().sum())


TableName         object
StateCode        float64
DistrictCode     float64
AreaName          object
BirthPlace        object
Total_Persons      int64
Total_Males        int64
Total_Females      int64
Rural_Persons      int64
Rural_Males        int64
Rural_Females      int64
Urban_Persons      int64
Urban_Males        int64
Urban_Females      int64
dtype: object 

TableName        1
StateCode        1
DistrictCode     1
AreaName         1
BirthPlace       0
Total_Persons    0
Total_Males      0
Total_Females    0
Rural_Persons    0
Rural_Males      0
Rural_Females    0
Urban_Persons    0
Urban_Males      0
Urban_Females    0
dtype: int64


In [5]:
num_cols = [c for c in df_raw.columns if c not in ('TableName','StateCode','DistrictCode','AreaName','BirthPlace')]
df_raw[num_cols].describe()


,Total_Persons,Total_Males,Total_Females,Rural_Persons,Rural_Males,Rural_Females,Urban_Persons,Urban_Males,Urban_Females
count,3.169000e+03,3.169000e+03,3.169000e+03,3.169000e+03,3.169000e+03,3.169000e+03,3.169000e+03,3.169000e+03,3.169000e+03
mean,3.056383e+06,1.573245e+06,1.483137e+06,2.104608e+06,1.079849e+06,1.024760e+06,9.517742e+05,4.933968e+05,4.583773e+05
std,4.071618e+07,2.150473e+07,1.940050e+07,2.860319e+07,1.520637e+07,1.360790e+07,1.224750e+07,6.398273e+06,5.858119e+06
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000e+01,4.000000e+00,5.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,7.000000e+00,3.000000e+00,4.000000e+00
50%,2.850000e+02,1.340000e+02,1.390000e+02,6.200000e+01,2.900000e+01,3.000000e+01,1.700000e+02,8.000000e+01,8.600000e+01
75%,7.786000e+03,3.860000e+03,3.841000e+03,2.152000e+03,9.690000e+02,1.062000e+03,4.485000e+03,2.256000e+03,2.197000e+03
max,1.210855e+09,6.232703e+08,5.875847e+08,8.337489e+08,4.277811e+08,4.059678e+08,3.771061e+08,1.954892e+08,1.816169e+08


## 2. Understanding subtotal rows


In [6]:
bp_series = df_raw['BirthPlace'].dropna().astype(str).str.strip()

SUMMARY_BP = [
    'Total Population','Born within India','Within the state of enumeration',
    'Born in the place of enumeration','Born elsewhere in the district of enumeration',
    'Born in other districts of the state',
    'States in India beyond the state of enumeration',
    'Born Outside India','Countries in Asia beyond India','Countries in Europe',
    'Countries in Africa','Countries in the Americas','Countries in Oceania',
    'Elsewhere','Unclassifiable'
]

print('BirthPlace values:')
for v in sorted(bp_series.unique()):
    n = (bp_series == v).sum()
    note = "  <- subtotal" if v in SUMMARY_BP else ("  <- header row" if v.isdigit() else "")
    print(f'  {n:>4}  {v}{note}')


BirthPlace values:
     1  1  <- header row
    36  Afganistan
    36  Andaman & Nicobar Islands
    36  Andhra Pradesh
    36  Arunachal Pradesh
    36  Assam
    36  Australia
    36  Bangladesh
    36  Bhutan
    36  Bihar
    36  Born Outside India  <- subtotal
    36  Born elsewhere in the district of enumeration  <- subtotal
    36  Born in other districts of the state  <- subtotal
    36  Born in the place of enumeration  <- subtotal
    36  Born within India  <- subtotal
    36  Canada
    36  Chandigarh
    36  Chhattisgarh
    36  China
    36  Countries in Africa  <- subtotal
    36  Countries in Asia beyond India  <- subtotal
    36  Countries in Europe  <- subtotal
    36  Countries in Oceania  <- subtotal
    36  Countries in the Americas  <- subtotal
    36  Dadra & Nagar Haveli
    36  Daman & Diu
   180  Elsewhere  <- subtotal
    36  Fiji
    36  France
    36  Germany
    36  Goa
    36  Gujarat
    36  Haryana
    36  Himachal Pradesh
    36  Indonesia
    36  Iran


In [7]:
an_series = df_raw['AreaName'].dropna().astype(str).str.strip()
print('AreaName values:')
for v in sorted(an_series.unique()):
    n = (an_series == v).sum()
    note = "  <- national total, remove" if v == "INDIA" else ""
    print(f'  {n:>4}  {v}{note}')


AreaName values:
    88  INDIA  <- national total, remove
    88  State - ANDAMAN & NICOBAR ISLANDS (35)
    88  State - ANDHRA PRADESH (28)
    88  State - ARUNACHAL PRADESH (12)
    88  State - ASSAM (18)
    88  State - BIHAR (10)
    88  State - CHANDIGARH (04)
    88  State - CHHATTISGARH (22)
    88  State - DADRA & NAGAR HAVELI (26)
    88  State - DAMAN & DIU (25)
    88  State - GOA (30)
    88  State - GUJARAT (24)
    88  State - HARYANA (06)
    88  State - HIMACHAL PRADESH (02)
    88  State - JAMMU & KASHMIR (01)
    88  State - JHARKHAND (20)
    88  State - KARNATAKA (29)
    88  State - KERALA (32)
    88  State - LAKSHADWEEP (31)
    88  State - MADHYA PRADESH (23)
    88  State - MAHARASHTRA (27)
    88  State - MANIPUR (14)
    88  State - MEGHALAYA (17)
    88  State - MIZORAM (15)
    88  State - NAGALAND (13)
    88  State - NCT OF DELHI (07)
    88  State - ODISHA (21)
    88  State - PUDUCHERRY (34)
    88  State - PUNJAB (03)
    88  State - RAJASTHAN (08)
   

## 3. Tier checks on India totals


In [8]:
india = df_raw[df_raw['AreaName'].astype(str).str.strip() == 'INDIA'].copy()
for c in ['Total_Persons', 'Rural_Persons', 'Urban_Persons']:
    india[c] = pd.to_numeric(india[c], errors='coerce')

def iv(bp_name):
    row = india[india['BirthPlace'].astype(str).str.strip() == bp_name]
    return float(row['Total_Persons'].values[0]) if len(row) else None

total = iv('Total Population')
b_india = iv('Born within India')
b_out = iv('Born Outside India')
w_state = iv('Within the state of enumeration')
beyond = iv('States in India beyond the state of enumeration')

print(f'Total Population : {total:>15,.0f}')
print(f'Born in India    : {b_india:>15,.0f}')
print(f'Born Outside     : {b_out:>15,.0f}')
print(f'Sum              : {b_india+b_out:>15,.0f}  ok = {abs(total - b_india - b_out) < 1}')
print()
print(f'Within state     : {w_state:>15,.0f}')
print(f'Beyond state     : {beyond:>15,.0f}')
print(f'Sum              : {w_state+beyond:>15,.0f}  ok = {abs(b_india - w_state - beyond) < 1}')


Total Population :   1,210,854,977
Born in India    :   1,205,201,066
Born Outside     :       5,363,099
Sum              :   1,210,564,165  ok = False

Within state     :   1,148,903,503
Beyond state     :      56,297,563
Sum              :   1,205,201,066  ok = True


## 4. Cleaning


In [9]:
df = df_raw.copy()

df = df[~df['BirthPlace'].astype(str).str.strip().str.match(r'^\d+$')]
df = df.dropna(subset=['AreaName', 'BirthPlace'])
df = df[df['AreaName'].astype(str).str.strip() != 'INDIA']

SUMMARY_BP = {
    'Total Population', 'Born within India',
    'Within the state of enumeration',
    'Born in the place of enumeration',
    'Born elsewhere in the district of enumeration',
    'Born in other districts of the state',
    'States in India beyond the state of enumeration',
    'Born Outside India', 'Countries in Asia beyond India',
    'Countries in Europe', 'Countries in Africa',
    'Countries in the Americas', 'Countries in Oceania',
    'Elsewhere', 'Unclassifiable',
}
df = df[~df['BirthPlace'].astype(str).str.strip().isin(SUMMARY_BP)]

num_cols = [c for c in df.columns if c not in ('TableName','StateCode','DistrictCode','AreaName','BirthPlace')]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

df = df[df['Total_Persons'] > 0]

df['AreaName'] = df['AreaName'].astype(str).str.replace(r'^State\s*-\s*', '', regex=True).str.replace(r'\s*\(\d+\)\s*$', '', regex=True).str.strip()
df['BirthPlace'] = df['BirthPlace'].astype(str).str.strip()

print(f'Rows: {len(df)}')
print(f"States: {df['AreaName'].nunique()},  BirthPlaces: {df['BirthPlace'].nunique()}")


Rows: 2029
States: 35,  BirthPlaces: 69


## 5. Validation checks


In [10]:
dupes = df.duplicated(subset=['AreaName', 'BirthPlace'])
print(f'Duplicate (state, birthplace) pairs: {dupes.sum()}')

mf_bad = (abs(df['Total_Persons'] - df['Total_Males'] - df['Total_Females']) > 1).sum()
print(f'Rows where Total != M+F: {mf_bad}')

ru_bad = (abs(df['Total_Persons'] - df['Rural_Persons'] - df['Urban_Persons']) > 1).sum()
print(f'Rows where Total != Rural+Urban: {ru_bad}')

sample = df['AreaName'].iloc[0]
raw_tot_row = df_raw[
    (df_raw['AreaName'].astype(str).str.contains(sample[:10], case=False, na=False)) &
    (df_raw['BirthPlace'].astype(str).str.strip() == 'Total Population')
]
if len(raw_tot_row):
    raw_tot = int(pd.to_numeric(raw_tot_row['Total_Persons'].values[0], errors='coerce'))
    clean_sum = df[df['AreaName'] == sample]['Total_Persons'].sum()
    print(f'\n{sample}: raw reported total = {raw_tot:,}, our cleaned sum = {clean_sum:,} ({clean_sum/raw_tot*100:.1f}%)')


Duplicate (state, birthplace) pairs: 0
Rows where Total != M+F: 0
Rows where Total != Rural+Urban: 0

JAMMU & KASHMIR: raw reported total = 12,541,302, our cleaned sum = 192,381 (1.5%)


## 6. Export cleaned file


In [ ]:
keep_cols = ['AreaName','BirthPlace','Total_Persons','Total_Males','Total_Females',
             'Rural_Persons','Rural_Males','Rural_Females',
             'Urban_Persons','Urban_Males','Urban_Females']
df_final = df[[c for c in keep_cols if c in df.columns]].reset_index(drop=True)

print(f'Shape  : {df_final.shape}')
print(f'Nulls  : {df_final.isnull().sum().sum()}')
print(f"Total  : {df_final['Total_Persons'].sum():,}")

df_final.to_csv('D01_cleaned.csv', index=False)
print('Saved -> D01_cleaned.csv')
df_final.head(10)
